# 🌲 Module 11: Multi-Temporal Change Detection Using Siamese Neural Networks

Welcome to **Module 11** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Multi-Temporal Imagery**: Understand bi-temporal change detection from pre-disturbance ($T_1$) and post-disturbance ($T_2$) satellite scenes.
2. **Siamese Neural Networks**: Implement weight-sharing dual-branch encoders ($E_\theta$) in PyTorch.
3. **Multi-Scale Differential Features**: Extract absolute difference representations $D^k = |f_1^k - f_2^k|$ across all hierarchical skip levels.
4. **Compound Change Loss**: Solve severe class imbalance in sparse deforestation ground truth masks.
5. **Benchmark vs Traditional $\Delta$NDVI**: Contrast Deep Siamese performance against traditional vegetation index differencing.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_11_siamese_change_detection.siamese_dataset import create_siamese_dataloaders
from modules.module_11_siamese_change_detection.siamese_model import SiameseUNetChangeDetector
from modules.module_11_siamese_change_detection.trainer import train_siamese_detector, evaluate_siamese_detector

%matplotlib inline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Module 11 Siamese environment ready on device: [{device}]")

## 1. Load Bi-Temporal Satellite Pairs

We load paired satellite imagery $(T_1, T_2)$ and ground-truth deforestation masks.

In [ ]:
train_loader, val_loader, test_loader = create_siamese_dataloaders(
    dataset_root="../dataset",
    tile_size=128,
    batch_size=8,
    max_train_tiles=32,
    max_val_tiles=8
)

print(f"Train pairs: {len(train_loader.dataset)} | Val pairs: {len(val_loader.dataset)} | Test pairs: {len(test_loader.dataset)}")

## 2. Inspect Siamese Architecture

Let's instantiate `SiameseUNetChangeDetector` and count parameters.

In [ ]:
model = SiameseUNetChangeDetector(in_channels=5, out_channels=1, base_features=16)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Siamese Network Trainable Parameters: {n_params:,}")

## 3. Train Siamese Model

We train the model using Compound Change Loss (Weighted BCE + Dice Loss).

In [ ]:
trained_model, history = train_siamese_detector(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=1e-3,
    device=device
)

## 4. Visualizing Multi-Temporal Change Predictions

Let's compare Pre-disturbance RGB ($T_1$), Post-disturbance RGB ($T_2$), Siamese Change Probability Map, and Ground Truth Deforestation.

In [ ]:
def make_rgb(tensor_5ch):
    arr = tensor_5ch.cpu().numpy()
    def strt(b):
        p2, p98 = np.percentile(b, (2, 98))
        return np.clip((b - p2) / (p98 - p2 + 1e-8), 0.0, 1.0) if p98 > p2 else np.clip(b, 0.0, 1.0)
    return np.stack([strt(arr[2]), strt(arr[1]), strt(arr[0])], axis=-1)

trained_model.eval()
sample_t1, sample_t2, sample_mask = next(iter(test_loader))
with torch.no_grad():
    logits, diff_feats = trained_model.forward_with_features(sample_t1[0:1].to(device), sample_t2[0:1].to(device))
    prob_map = torch.sigmoid(logits).cpu().numpy()[0, 0]
    bin_pred = (prob_map >= 0.5).astype(int)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(make_rgb(sample_t1[0]))
axes[0].set_title("Pre-Disturbance (T1)")
axes[0].axis("off")

axes[1].imshow(make_rgb(sample_t2[0]))
axes[1].set_title("Post-Disturbance (T2)")
axes[1].axis("off")

im_prob = axes[2].imshow(prob_map, cmap="inferno", vmin=0, vmax=1)
axes[2].set_title("Siamese Change Prob")
axes[2].axis("off")
plt.colorbar(im_prob, ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(bin_pred, cmap="Reds", vmin=0, vmax=1)
axes[3].set_title("Binary Deforestation")
axes[3].axis("off")

axes[4].imshow(sample_mask[0, 0].numpy(), cmap="Greens", vmin=0, vmax=1)
axes[4].set_title("Ground Truth Mask")
axes[4].axis("off")

plt.tight_layout()
plt.show()